# ETL: beta-convergence, 2004-2024

The notebook explores the dataset, it does not define it: the download lives in
`src/fetch_data.py` and the transformations in `src/etl.py`. Inference is in
`R/regressions.R`.

In [1]:
import sys
import numpy as np
import pandas as pd
import statsmodels.api as sm

sys.path.append("../src")

from etl import build_dataset, sigma_convergence # etl.py

data = build_dataset()
print(f"{len(data)} economies | dropped: {data.attrs['dropped_incomplete']}")
data.head(3)

153 economies | dropped: 6


,country_code,country_name,region,income_group,group,developed,population,log_y0_pre_crisis,log_y0_recuperation,log_y0_stability,...,gdp_pc_2015,gdp_pc_2016,gdp_pc_2017,gdp_pc_2018,gdp_pc_2019,gdp_pc_2020,gdp_pc_2021,gdp_pc_2022,gdp_pc_2023,gdp_pc_2024
0,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,Emerging,0,42647492,5.824930,6.034637,6.364069,...,565.569730,563.872337,562.769574,553.125152,557.861533,527.834554,408.625855,377.665627,378.066303,374.376696
1,ALB,Albania,Europe & Central Asia,Upper middle income,Emerging,0,2377128,7.850609,8.115515,8.267680,...,4199.539129,4431.556358,4648.230787,4893.823755,5072.436484,4980.616995,5511.757434,5867.650962,6204.435256,6549.233874
2,DZA,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,Emerging,0,46814308,8.310484,8.381961,8.421394,...,4685.059027,4768.731401,4742.900755,4717.003589,4672.664087,4363.685338,4456.746876,4544.466881,4660.405457,4765.729002


Every World Bank member economy has at least one million inhabitants, and reports GDP per capita in every year the periods need.

In [2]:
print(data.groupby(["group", "income_group"]).size())
print()
print(data.groupby("group")["population"].describe()[["count", "min", "50%", "max"]])

group      income_group       
Developed  High income            50
Emerging   Low income             20
           Lower middle income    40
           Upper middle income    43
dtype: int64

           count        min         50%           max
group                                                
Developed   50.0  1358282.0   9091782.0  3.400038e+08
Emerging   103.0  1242822.0  18135478.0  1.450936e+09


In [3]:
analysis_cols = [c for c in data.columns if c.startswith(("growth_", "log_y0_"))]
print(data[analysis_cols].isna().mean().to_string())
data[analysis_cols].describe().transpose().round(3)

log_y0_pre_crisis      0.0
log_y0_recuperation    0.0
log_y0_stability       0.0
log_y0_recent          0.0
log_y0_full            0.0
growth_pre_crisis      0.0
growth_recuperation    0.0
growth_stability       0.0
growth_recent          0.0
growth_full            0.0


,count,mean,std,min,25%,50%,75%,max
log_y0_pre_crisis,153.0,8.298,1.507,5.553,7.083,8.182,9.311,11.238
log_y0_recuperation,153.0,8.445,1.492,5.545,7.166,8.472,9.503,11.338
log_y0_stability,153.0,8.532,1.438,5.558,7.244,8.519,9.632,11.338
log_y0_recent,153.0,8.619,1.429,5.589,7.382,8.587,9.753,11.392
log_y0_full,153.0,8.298,1.507,5.553,7.083,8.182,9.311,11.238
growth_pre_crisis,153.0,3.677,3.246,-9.313,1.674,3.297,5.265,20.417
growth_recuperation,153.0,1.746,2.668,-6.089,0.144,1.803,3.329,10.651
growth_stability,153.0,1.727,2.693,-13.537,0.829,1.727,3.370,8.695
growth_recent,153.0,0.906,2.505,-11.967,-0.048,1.219,2.291,5.626
growth_full,153.0,1.875,1.867,-4.341,0.735,1.803,3.063,7.169


## The regressor that matters

Growth on past growth and growth on initial income are nearly unrelated in this
sample, so the choice of regressor changes the answer rather than refining it.

In [4]:
print("corr(growth 2008-2013, growth 2004-2008) = "
      f"{data['growth_recuperation'].corr(data['growth_pre_crisis']):.3f}")
print("corr(growth 2008-2013, log y0 2008)      = "
      f"{data['growth_recuperation'].corr(data['log_y0_recuperation']):.3f}")

corr(growth 2008-2013, growth 2004-2008) = 0.253
corr(growth 2008-2013, log y0 2008)      = -0.441


In [5]:
model = sm.OLS(data["growth_full"],
               sm.add_constant(data["log_y0_full"])).fit(cov_type="HC1")
print(model.summary().tables[1])

beta = model.params["log_y0_full"] / 100
lam = -np.log(1 + beta * 20) / 20
print()
print(f"lambda = {lam:.3%}/yr, half-life = {np.log(2) / lam:.0f} years")

                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           4.8119      0.744      6.467      0.000       3.354       6.270
log_y0_full    -0.3539      0.083     -4.277      0.000      -0.516      -0.192

lambda = 0.367%/yr, half-life = 189 years


## Sigma-convergence

Whether the income distribution narrows, which beta-convergence alone
does not guarantee.

In [6]:
dispersion = sigma_convergence(data)
dispersion.pivot(index="year", columns="group", values="sd_log_gdp_pc").round(3)

group,All,Developed,Emerging
year,,,
2004,1.507,0.725,0.953
2005,1.505,0.708,0.961
2006,1.506,0.690,0.970
2007,1.503,0.665,0.979
2008,1.492,0.643,0.981
2009,1.466,0.647,0.964
2010,1.460,0.650,0.963
2011,1.456,0.644,0.957
2012,1.446,0.635,0.963
